# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step exploration and processing of the FAIR² dataset using the `mlcroissant` library. All dataset entities (record sets, fields, columns, etc.) are referenced by their `@id` fields to ensure clarity and reproducibility.

### Dataset Source
The dataset is defined by a Croissant schema, accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is available
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id` values to discover the schema and data structure. 

In [ ]:
# List available record sets and their IDs
record_sets = dataset.metadata.record_sets
print("Available record sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name'] if 'name' in rs else 'No name'}")

# For each record set, list its fields and columns by @id
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name', 'No name')})")
    fields = rs['field'] if 'field' in rs else []
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields and columns (@id):")
    for field in fields:
        print(f"    - Field @id: {field['@id']}")
        if 'column' in field:
            columns = field['column']
            if isinstance(columns, dict):
                columns = [columns]
            for col in columns:
                print(f"      - Column @id: {col['@id']}")

## 3. Data Extraction
Load data from specific record set(s) into DataFrames. Use the record set `@id`s and field `@id`s found above to access the data.

In [ ]:
# Select record sets for analysis (by @id). Choose all for demonstration:
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows. Columns: {df.columns.tolist()}")
    print(df.head(), "\n")

## 4. Exploratory Data Analysis (EDA)
Apply typical steps: filtering, normalization, and grouping, using field and column `@id`s. 

- Select one numeric field (`@id`) for outlier removal and normalization.
- Select one group field (`@id`) for grouping. 

> **Note**: Replace the example field `@id`s below with those that are relevant for your analysis from the overview above.

In [ ]:
# Choose one record set to analyze (replace with your main data table's @id):
main_record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[main_record_set_id].copy() if main_record_set_id else pd.DataFrame()

# Select a numeric field (by column @id from previous step, e.g. 'cr:coveragePercent')
# Pick from df.columns, for example:
numeric_field_id = None
for col in df.columns:
    if "coverage" in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    try:
        # Default to the first column with numeric type
        numeric_field_id = df.select_dtypes(include=['float','int']).columns[0]
    except:
        numeric_field_id = df.columns[0]

print(f"Using numeric field: {numeric_field_id}")

# Threshold for filtering
try:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
except:
    threshold = 0

# Filter records with numeric_field > threshold
filtered_df = df.copy()
if numeric_field_id in df.columns:
    try:
        filtered_df = df[df[numeric_field_id] > threshold]
    except:
        pass
print(f"Filtered records with {numeric_field_id} > {threshold} (showing top 5):")
print(filtered_df.head())

# Normalize the numeric field (Z-score normalization)
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nSample normalized field:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field (pick a textual/categorical field, e.g. 'cr:sampleType')
group_field_id = None
for col in df.columns:
    if (col != numeric_field_id) and (df[col].dtype == object or pd.api.types.is_categorical_dtype(df[col])):
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset using the normalized numeric field and grouping field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot by group (if applicable)
if group_field_id:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} distribution grouped by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
*In this notebook, we used `mlcroissant` to load and analyze the FAIR² dataset. We:

- Explored dataset metadata and data structure via Croissant schema.
- Listed all record sets and their fields/columns with `@id` references for reproducibility.
- Loaded data dynamically using only entity `@id`s.
- Performed exploratory data analysis and simple data processing tasks.
- Visualized numeric field distributions and their groupings.

You can now extend this notebook for more advanced statistics or ML experiments using any field by referencing its `@id`.
